# Loading

In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import pandas as pd
import scipy.sparse as sp
import time
import os
import seaborn as sns
import matplotlib.pyplot as plt
import json

In [ ]:
from matplotlib import rcParams
import matplotlib as mpl
import matplotlib.font_manager as fm
import tempfile
import requests

rcParams["figure.dpi"] = 300
rcParams["savefig.dpi"] = 300
rcParams["figure.facecolor"] = 'white'
rcParams["axes.facecolor"] = 'white'

global _vector_friendly
_vector_friendly = True

font_path = 'arial.ttf'

fm.fontManager = fm.FontManager()
            
# 2) Add your file
fm.fontManager.addfont(font_path)

# 3) Now find out what name it uses
name = fm.FontProperties(fname=font_path).get_name()
print("Registered as:", name)

# 4) Point rcParams at that name
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = [name, 'DejaVu Sans']

plt.rcParams['axes.grid'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42

# Data process

In [ ]:
# Jupyter cell: 计算每个子文件夹中的 GRN（regulon_dict）对应 TF 的 eigenvector centrality，
# 并返回 spatial domain × TF 的 DataFrame（缺失值填 0）。
from pathlib import Path
import scanpy as sc
import pandas as pd
from igraph import Graph
from typing import Dict, Iterable
from tqdm import tqdm

def _compute_eigenvector_from_regulon_dict(regulon_dict: Dict[str, Iterable[str]],
                                           weight_value: float = 1.0) -> Dict[str, float]:
    """
    输入 regulon_dict: TF -> iterable(target genes)
    返回 node_name -> eigenvector centrality（igraph计算，directed=False，weights='weight'）
    若图为空返回 {}。
    """
    edges = []
    for tf, targets in regulon_dict.items():
        for t in targets:
            edges.append((str(tf), str(t)))
    if len(edges) == 0:
        return {}
    # 用 TupleList 保留顶点名，先建立有向图（与原CellOracle一致），
    # 计算时指定 directed=False 即把方向忽略
    g = Graph.TupleList(edges, directed=True, vertex_name_attr="name")
    # 统一权重
    g.es["weight"] = [float(weight_value)] * len(g.es)
    ev = g.eigenvector_centrality(directed=False, weights="weight")
    names = g.vs["name"]
    return dict(zip(names, ev))
    
def build_spatial_vs_grn_matrix(root_folder: str = "GRN_GPU_output") -> pd.DataFrame:
    """
    遍历 root_folder 的一级子文件夹，寻找子文件夹下名字为 grn_{subfolder}_spagrn.h5ad 的文件并读取。
    返回 DataFrame：index = 子文件夹名 (n=GRN数量)，columns = 所有出现过的 TF（regulon_dict.keys()），
    值为该 TF 在该 domain 中的 eigenvector centrality（若不存在则为 0）。
    """
    root = Path(root_folder)
    if not root.exists():
        raise FileNotFoundError(f"Root folder not found: {root_folder}")
    
    domain_names = []
    per_domain_tf_scores = {}
    all_tfs = set()

    # 遍历一级子文件夹
    for sub in tqdm(sorted([p for p in root.iterdir() if p.is_dir()])):
        domain = sub.name
        expected_fname = f"tmp_files/regulons.json"
        expected_path = sub / expected_fname
        
        if not expected_path.exists():
            # 没有精确匹配，跳过该子文件夹
            continue
            
        # 读取 json
        try:
            with open(expected_path, 'r', encoding='utf-8') as file:
                grn_dict = json.load(file)
        except Exception as e:
            print(f"Warning: failed to read {expected_path}: {e}")
            continue

        regulon = grn_dict
        # 确保 regulon 是 dict-like
        if regulon is None:
            regulon = {}
            
        # --- (关键修改 1: 获取GRN数量并格式化新名称) ---
        grn_count = len(regulon)
        formatted_domain_name = f"{domain} (n={grn_count})"
        # ---------------------------------------------

        # 计算节点（包括 TF 与 targets）级别的 eigenvector centrality
        node_scores = _compute_eigenvector_from_regulon_dict(regulon, weight_value=1.0)
        
        # 我们只在矩阵中放置 TF（regulon_dict 的 keys）
        tf_scores = {}
        for tf in regulon.keys():
            tf_str = str(tf)
            score = float(node_scores.get(tf_str, 0.0))
            tf_scores[tf_str] = score
            all_tfs.add(tf_str)
            
        # --- (关键修改 2: 使用格式化的新名称) ---
        domain_names.append(formatted_domain_name)
        per_domain_tf_scores[formatted_domain_name] = tf_scores
        # ----------------------------------------

    # 构建 DataFrame（域为行，TF 为列），缺失填 0
    sorted_tfs = sorted(all_tfs)
    df = pd.DataFrame(index=domain_names, columns=sorted_tfs, dtype=float).fillna(0.0)
    
    for domain_name_formatted, tf_scores in per_domain_tf_scores.items():
        for tf, score in tf_scores.items():
            df.at[domain_name_formatted, tf] = score
            
    df = df.sort_index()
    return df

In [ ]:
# ---------- min-max normalization per column (TF) 0-1 ----------
def minmax_normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    对 df 的每一列做 0-1 标准化 (min-max)。
    若列常数（max == min），则整个列设为 0 (或你可以选择设为 0.0)。
    """
    df_norm = df.copy().astype(float)
    col_min = df_norm.min(axis=0)
    col_max = df_norm.max(axis=0)
    denom = (col_max - col_min).replace(0, np.nan)
    df_norm = (df_norm - col_min) / denom
    # 将原本 denom==0 的列填 0
    df_norm = df_norm.fillna(0.0)
    return df_norm

In [ ]:
root_folder = "Output/GRN_GPU_output"  
df_domain_vs_tf = build_spatial_vs_grn_matrix(root_folder)

print("Result D:", df_domain_vs_tf.shape)
display(df_domain_vs_tf.head())

# Visualization

In [ ]:
# 归一化（对每个 TF 的列进行 0-1 min-max）
df_norm = minmax_normalize_columns(df_domain_vs_tf)
print("Normalized matrix (0-1) — sample:")
display(df_norm.head())

# ---------- 聚类热图可视化 ----------
# 配置可视化风格
sns.set(context="notebook", style="white")
# 使用 seaborn.clustermap：对行和列都进行层次聚类（默认）
# 你可以调整 method、metric、col_cluster/row_cluster 等参数
cg = sns.clustermap(df_norm,
                    cmap="viridis",
                    figsize=(12, 10),
                    method="average",    # linkage method
                    metric="euclidean",  # distance metric
                    standard_scale=None, # 我们已自行归一化，不需要 seaborn 再标准化
                    z_score=None,
                    row_cluster=True,
                    col_cluster=True)

plt.suptitle("Clustermap of TF eigenvector centrality (columns min-max normalized per TF)", y=1.02)
plt.show()


In [ ]:
# 在 Jupyter 中运行：单样本 t 检验（留一法）优先，sd=0 时退回经验秩法，
# 为每个基因分配显著的 region（p<alpha），否则标 'not significant'。
import numpy as np
import pandas as pd
from scipy import stats

# 检查 df_norm 是否存在
try:
    df_norm
except NameError:
    raise NameError("未找到 df_norm。请先生成归一化矩阵并命名为 df_norm（index=regions, columns=genes）。")

def assign_region_by_leave_one_ttest(df_norm, alpha=0.05, alternative='greater', fallback_to_empirical=True):
    """
    对每个基因（列），对每个 region 做 leave-one-out t-test:
      - 对 region i，x = value_i, others = values without i
      - 若 sd(others) > 0，计算 t = (x - mean(others)) / sd(others), df = n-2
        然后取单侧 p（'greater'：P(T >= t)，'less'：P(T <= t)，'two-sided'：2*P(T >= |t|)）
      - 若 sd(others) == 0 或 t 检验不可计算，则退回经验秩法（若 fallback_to_empirical=True）
    最终为每个基因选取满足 p < alpha 的 region 中 p 最小者；若没有则标 'not significant'。
    返回 DataFrame index=gene columns=['region','p_value','method']。
    """
    genes = df_norm.columns.tolist()
    regions = df_norm.index.tolist()
    n_regions = len(regions)
    results = []

    for g in genes:
        col = df_norm[g].values.astype(float)
        p_vals = np.ones(n_regions, dtype=float)
        method_used = np.array([''] * n_regions, dtype=object)

        for i in range(n_regions):
            x = col[i]
            others = np.delete(col, i)
            # handle NaNs
            others = others[~np.isnan(others)]
            if others.size < 2:
                # not enough data to compute sd reliably -> fallback
                sd_others = 0.0
            else:
                sd_others = np.std(others, ddof=1)  # sample std

            if sd_others > 0 and not np.isnan(sd_others):
                mean_others = np.mean(others)
                # t statistic: (x - mean_others) / (sd_others)
                t_stat = (x - mean_others) / sd_others
                dfree = len(others) - 1  # = n_regions - 2 typically
                # compute p according to alternative
                if alternative == 'greater':
                    # P(T >= t_stat)
                    p = stats.t.sf(t_stat, df=dfree)
                elif alternative == 'less':
                    p = stats.t.cdf(t_stat, df=dfree)
                elif alternative == 'two-sided':
                    p = 2.0 * stats.t.sf(abs(t_stat), df=dfree)
                else:
                    raise ValueError("alternative must be 'greater', 'less' or 'two-sided'")
                p_vals[i] = float(p)
                method_used[i] = 't_test'
            else:
                # fallback empirical rank if allowed
                if fallback_to_empirical:
                    if alternative == 'greater':
                        p = np.sum(col >= x) / float(n_regions)
                    elif alternative == 'less':
                        p = np.sum(col <= x) / float(n_regions)
                    elif alternative == 'two-sided':
                        med = np.median(col)
                        p = np.sum(np.abs(col - med) >= abs(x - med)) / float(n_regions)
                    else:
                        raise ValueError("alternative must be 'greater', 'less' or 'two-sided'")
                    p_vals[i] = float(p)
                    method_used[i] = 'empirical_rank'
                else:
                    # leave p as 1.0 and method empty
                    p_vals[i] = 1.0
                    method_used[i] = 'insufficient_data'

        # find significant regions
        sig_idx = np.where(p_vals < alpha)[0]
        if sig_idx.size == 0:
            results.append((g, 'not significant', np.nan, 'none'))
        else:
            # choose smallest p among significant ones (if tie, smallest p's first occurrence)
            min_idx = sig_idx[np.argmin(p_vals[sig_idx])]
            results.append((g, regions[min_idx], float(p_vals[min_idx]), method_used[min_idx]))

    labels_df = pd.DataFrame(results, columns=['gene', 'region', 'p_value', 'method']).set_index('gene')
    return labels_df


In [ ]:
labels_df = assign_region_by_leave_one_ttest(df_norm, alpha=0.05, alternative='greater', fallback_to_empirical=True)
labels_df.to_csv("Output/GRN_GPU_output/GRN_summary.csv")
labels_df

In [ ]:
set(labels_df['region'])

In [ ]:
adata_grn = ad.AnnData(df_norm.T)
adata_grn.obs['region'] = labels_df.loc[adata_grn.obs_names,'region']

desired_order = ['CP_SP (n=233)','Di_ChP (n=274)','HIP (n=261)','Hindbrain (n=265)','Hindbrain_ChP (n=290)',
                 'IZ (n=233)','LGE (n=248)','MGE (n=241)','NAc (n=269)','PONS (n=246)','STR (n=237)','SVZ (n=241)',
                 'VZ (n=263)', 'VZ_GE (n=298)','amygdala (n=269)','cerebellum (n=281)',
                 'ependymal_region (n=301)','midbrain (n=234)','subthalamic_nucleus (n=300)',
                'thalamus (n=253)','not significant']
adata_grn.obs['region'] = adata_grn.obs['region'].astype('category')
adata_grn.obs['region'] = adata_grn.obs['region'].cat.reorder_categories(desired_order, ordered=False)
ax = sc.pl.heatmap(
    adata_grn,
    adata_grn.var_names,
    groupby="region",
    vmin=0,
    vmax=2,
    cmap="Reds",
    dendrogram=False,
    swap_axes=True,
    figsize=(11, 4),
)

In [ ]:
labels_df = assign_region_by_leave_one_ttest(df_norm, alpha=0.05, alternative='greater', fallback_to_empirical=True)
labels_df = labels_df[labels_df['region']!='not significant']
labels_df

In [ ]:
for region in set(labels_df['region']):
    number = len(labels_df[labels_df['region']==region].index)
    print(f'TF in region {region} is {number}')

In [ ]:
adata_grn = ad.AnnData(df_norm.T)
adata_grn = adata_grn[labels_df.index,:]
adata_grn.obs['region'] = labels_df.loc[adata_grn.obs_names,'region']

desired_order = ['CP_SP (n=233)','Di_ChP (n=274)','HIP (n=261)','Hindbrain (n=265)','Hindbrain_ChP (n=290)',
                 'IZ (n=233)','LGE (n=248)','MGE (n=241)','NAc (n=269)','PONS (n=246)','STR (n=237)','SVZ (n=241)',
                 'VZ (n=263)', 'VZ_GE (n=298)','amygdala (n=269)','cerebellum (n=281)',
                 'ependymal_region (n=301)','midbrain (n=234)','subthalamic_nucleus (n=300)',
                'thalamus (n=253)']
adata_grn.obs['region'] = adata_grn.obs['region'].astype('category')
adata_grn.obs['region'] = adata_grn.obs['region'].cat.reorder_categories(desired_order, ordered=False)

In [ ]:
cluster2annotation = {
    'CP_SP (n=233)': 'CP_SP (n=19)',
    'Di_ChP (n=274)': 'Di_ChP (n=40)',
    'HIP (n=261)': 'HIP (n=22)',
    'Hindbrain (n=265)': 'Hindbrain (n=23)',
    'Hindbrain_ChP (n=290)': 'Hindbrain_ChP (n=37)',
    'IZ (n=233)': 'IZ (n=22)',
    'LGE (n=248)': 'LGE (n=33)',
    'MGE (n=241)': 'MGE (n=27)',
    'NAc (n=269)': 'NAc (n=24)',
    'PONS (n=246)': 'PONS (n=23)',
    'STR (n=237)': 'STR (n=39)',
    'SVZ (n=241)': 'SVZ (n=33)',
    'VZ (n=263)': 'VZ (n=32)',
    'VZ_GE (n=298)': 'VZ_GE (n=32)',
    'amygdala (n=269)': 'amygdala (n=34)',
    'cerebellum (n=281)': 'cerebellum (n=51)',
    'ependymal_region (n=301)': 'ependymal_region (n=40)',
    'midbrain (n=234)': 'midbrain (n=45)',
    'subthalamic_nucleus (n=300)': 'subthalamic_nucleus (n=76)',
    'thalamus (n=253)': 'thalamus (n=39)'
}

adata_grn.obs['region'] = adata_grn.obs['region'].map(cluster2annotation).astype('category')
adata_grn.var_names = ['CP_SP (n=19)','Di_ChP (n=40)','HIP (n=22)','Hindbrain (n=23)','Hindbrain_ChP (n=37)','IZ (n=22)',
                      'LGE (n=33)','MGE (n=27)','NAc (n=24)','PONS (n=23)','STR (n=39)','SVZ (n=33)','VZ (n=32)',
                      'VZ_GE (n=32)','amygdala (n=34)','cerebellum (n=51)','ependymal_region (n=40)','midbrain (n=45)',
                      'subthalamic_nucleus (n=76)','thalamus (n=39)']

In [ ]:
adata_grn.var_names

In [ ]:
ax = sc.pl.heatmap(
    adata_grn,
    adata_grn.var_names,
    groupby="region",
    vmin=0,
    vmax=1,
    cmap="GnBu",
    dendrogram=False,
    swap_axes=True,
    show=False,
    figsize=(8, 4.5),
)
plt.savefig("Figure/whole_brain_selected_GRN/region_GRN_heatmap.pdf", dpi=300, bbox_inches = "tight")